# AERIS — Pneumonie Edge v3, v1 etalonne sur le meme hold-out

Dataset images : `kaggle-pneumonia-chest-xray`
Modeles v1 : `boubaniang/aeris-modeles-v1` (dataset prive, quatre `.h5` en service)

## Les deux defauts de fond corriges par rapport a v2

1. **Architecture.** v2 (3 convolutions puis `GlobalAveragePooling2D`) compte
   ~25 000 parametres et ecrase la geometrie spatiale avant les Dense. Le modele
   v1 en service en compte **a confirmer en execution** et utilise `Flatten`. Ce kernel
   retablit l'architecture v1 exacte, sortie comprise, donc **specification
   identique au service** (`taille 150, canaux L, rescale`) et **aucune modification du service**.

2. **Fuite de donnees.** Les manifestes v2 separees par chemin laissent
   `Tuberculosis-243.png` (test) identique a `Tuberculosis-244.png` (train),
   meme SHA-256. Ici le split est construit **par regroupement SHA-256** : deux
   images de contenu identique ne peuvent pas atterrir dans deux lots differents,
   et le controle est refait a l'execution, avec arret du kernel si une fuite
   subsiste.

## V1 et V3 sont mesures dans la meme execution

Aucun score v1 n'etait disponible pour la tuberculose : les comparaison v1/v2
existantes ne portent que sur la pneumonie. Ce kernel evalue donc **le modele v1
et le modele v3 sur le meme hold-out**, avec la meme augmentation, dans la meme
execution. La comparaison est directe et dit si v3 vaut un remplacement.

L'augmentation est faite avec **Pillow**, pas `tf.image` : sur TensorFlow
2.20, `tf.image.random_rotate` et `tf.image.random_zoom` n'existent plus. Deux
versions de kernel ont ete perdues la-dessus avant d'identifier la cause.


In [ ]:
print('=== cellule : imports et configuration ===', flush=True)

# Pile unique et obligatoire. Les `.h5` du service sont des artefacts Keras 2 :
# la pile Keras 3 ne sait pas les deserialiser (`Could not locate class
# 'Sequential'`), et un repli silencieux produirait un modele reconstruit avec
# des poids non verifies.(tf_keras = Keras 2.20, livre avec TensorFlow.)
try:
    import tf_keras
except ImportError as _err:
    raise SystemExit(
        'tf_keras (Keras 2) est indispensable pour lire les modeles v1 du '
        'service ; la pile Keras 3 ne sait pas les deserialiser.'
    ) from _err

keras = tf_keras
print('pile de travail :', 'tf_keras', keras.__version__, flush=True)
import hashlib
import json

import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (150, 150)
COLOR_MODE = 'grayscale'
BATCH = 32
EPOCHS = 60
PATIENCE = 12
EPS = 1e-7
PART_TEST = 0.10
PART_VALIDATION = 0.10

WORK = Path('/kaggle/working')
RACINE_INPUT = Path('/kaggle/input/datasets')
ART = WORK / 'artifacts_pn_edge_v3'
ART.mkdir(parents=True, exist_ok=True)


BASES = [RACINE_INPUT, Path('/kaggle/input'), Path('/kaggle/working'), Path('.')]


def trouver_repertoire(noms):
    # Cherche un nom de dossier interne connu, recursivement, sur plusieurs bases.
    # Kaggle retire le prefixe proprietaire du dossier d'entree : reconstruire le
    # chemin depuis le slug du dataset est fragile, et la convention a change entre
    # les versions de l'image. Chercher un dossier interne au jeu donnees rend la
    # resolution independante de cette convention.
    if isinstance(noms, str):
        noms = [noms]
    for base in BASES:
        if not base.exists():
            continue
        for nom in noms:
            for candidat in sorted(base.glob(f'**/{nom}')):
                if candidat.is_dir():
                    return candidat
    listing = []
    for base in BASES:
        if base.exists():
            listing.extend(sorted(p.name for p in base.iterdir() if p.is_dir()))
    raise FileNotFoundError(f'aucun dossier parmi {noms} ; repertoires presents : {listing}')


DATA = trouver_repertoire(['chest_xray', 'chest-xray-pneumonia', 'kaggle-pneumonia-chest-xray'])
print('dataset resolu :', DATA, flush=True)
print('tensorflow', tf.__version__, flush=True)
print('dataset', DATA, flush=True)
print('taille du dataset en Mio', round(sum(p.stat().st_size for p in DATA.rglob('*') if p.is_file()) / 2**20, 1), flush=True)


In [ ]:
print('=== cellule : indexation et empreintes ===', flush=True)
CLASSES = {'NORMAL': 0, 'PNEUMONIA': 1}

MOTIFS = ('*.png', '*.jpg', '*.jpeg', '*.bmp', '*.PNG', '*.JPG', '*.JPEG')




def dossiers_de_classe(classe):
    # Tous les dossiers portant ce nom de classe. Le jeu Mooney range les images
    # dans chest_xray/{train,test}/<classe> : ne garder que DATA / classe ne
    # reverrait que la moitie du corpus, alors que la comparaison v1/v2 anterieure
    # portait sur les 624 images de test. On prend donc tous les dossiers trouves.
    direct = DATA / classe
    if direct.is_dir():
        return [direct]
    trouves = [
        candidat for candidat in sorted(DATA.rglob('*'))
        if candidat.is_dir()
        and candidat.name.lower() == classe.lower()
        and not any(part in IGNORES for part in candidat.parts)
    ]
    if not trouves:
        raise FileNotFoundError(f'classe absente : {classe} sous {DATA}')
    # Ne garder que les dossiers les plus superficiels : le jeu Mooney embarque
    # une seconde copie complete de l'arborescence (`chest_xray/chest_xray/...`)
    # et une copie `__MACOSX`. Les compter revient a doubler le corpus, donc a
    # Interpreter deux fois la meme image comme deux echantillons.
    retenus = [
        candidat for candidat in trouves
        if not any(autre != candidat and candidat.is_relative_to(autre) for autre in trouves)
    ]
    return retenus


IGNORES = ('__MACOSX', '.ipynb_checkpoints', '.git')


def collecter_images(dossiers):
    # Sur un systeme de fichiers insensible a la casse, `*.png` et `*.PNG`
    # designent les memes fichiers : sans deduplication par chemin resolu, chaque
    # image est comptee deux fois et le corpus parait deux fois plus grand.
    vus = set()
    retenus = []
    for dossier in dossiers:
        for motif in MOTIFS:
            for chemin in sorted(dossier.glob(motif)):
                resolu = chemin.resolve()
                if resolu in vus:
                    continue
                if any(part in IGNORES for part in resolu.parts):
                    continue
                vus.add(resolu)
                retenus.append(chemin)
    return retenus


lignes = []
for classe, etiquette in CLASSES.items():
    dossiers = dossiers_de_classe(classe)
    motifs = collecter_images(dossiers)
    for dossier in dossiers:
        print('  dossier', dossier, flush=True)
    print(classe, len(dossiers), 'dossier(s)', len(motifs), 'images distinctes', flush=True)
    for chemin in motifs:
        lignes.append({
            'chemin': str(chemin),
            'chemin_relatif': chemin.relative_to(DATA).as_posix(),
            'etiquette': etiquette,
        })

cadre = pd.DataFrame(lignes)
print('total', len(cadre), 'positifs', int(cadre['etiquette'].sum()), flush=True)
if len(cadre) == 0:
    raise SystemExit('aucune image trouvee : verifier la structure du dataset')


def empreinte(chemin):
    hachage = hashlib.sha256()
    with open(chemin, 'rb') as f:
        for bloc in iter(lambda: f.read(1 << 20), b''):
            hachage.update(bloc)
    return hachage.hexdigest()


print('calcul des SHA-256...', flush=True)
cadre['sha256'] = [empreinte(c) for c in cadre['chemin']]
print('empreintes calculees', len(cadre), flush=True)

avant_dedup = len(cadre)
cadre = cadre.drop_duplicates(subset='sha256', keep='first').reset_index(drop=True)
print('lignes', avant_dedup, '->', len(cadre), 'apres deduplication par contenu', flush=True)

coherents = cadre.groupby('sha256')['etiquette'].nunique()
ambigus = coherents[coherents > 1]
if len(ambigus) > 0:
    detail = cadre[cadre['sha256'].isin(ambigus.index)].sort_values('sha256')
    print(cadre.groupby('etiquette').size(), flush=True)
    raise SystemExit(
        f'{len(ambigus)} contenus portent deux etiquettes differentes ; '
        f'exemples : {detail.groupby("sha256")["etiquette"].apply(list).head(5).to_dict()}'
    )
print('contenus distincts', len(coherents), flush=True)
print('images dupliquees', int(cadre['sha256'].duplicated(keep=False).sum()), flush=True)


In [ ]:
print('=== cellule : split par regroupement de contenu ===', flush=True)


def split_par_contenu(cadre, graine):
    """Attribue chaque SHA-256 a un seul lot, en respectant 80/10/10 par classe."""
    groupes = (
        cadre.groupby('sha256')
        .agg(etiquette=('etiquette', 'first'), taille=('etiquette', 'size'))
        .reset_index()
    )
    groupes = groupes.sample(frac=1.0, random_state=graine).reset_index(drop=True)
    attribution = {}
    for etiquette in sorted(groupes['etiquette'].unique()):
        sous = groupes[groupes['etiquette'] == etiquette]
        total = float(sous['taille'].sum())
        cible_test = total * PART_TEST
        cible_validation = total * PART_VALIDATION
        cumule_test = 0.0
        cumule_validation = 0.0
        for _, groupe in sous.iterrows():
            sha = groupe['sha256']
            if cumule_test < cible_test:
                attribution[sha] = 'test'
                cumule_test += float(groupe['taille'])
            elif cumule_validation < cible_validation:
                attribution[sha] = 'validation'
                cumule_validation += float(groupe['taille'])
            else:
                attribution[sha] = 'entrainement'
    cadre = cadre.copy()
    cadre['lot'] = cadre['sha256'].map(attribution)
    return cadre


cadre = split_par_contenu(cadre, SEED)
noms = ('entrainement', 'validation', 'test')
lots = {}
for nom in noms:
    sous = cadre[cadre['lot'] == nom].reset_index(drop=True)
    lots[nom] = sous
    print(nom, len(sous), 'images,', int(sous['etiquette'].sum()), 'positifs', flush=True)

ensembles = {nom: set(lots[nom]['sha256'].tolist()) for nom in noms}
fuite_totale = 0
for i, nom_a in enumerate(noms):
    for nom_b in noms[i + 1:]:
        communs = ensembles[nom_a].intersection(ensembles[nom_b])
        fuite_totale += len(communs)
        print('fuite', nom_a, '/', nom_b, ':', len(communs), flush=True)
        if communs:
            raise SystemExit(f'fuite SHA-256 entre {nom_a} et {nom_b} : {len(communs)} contenus')
print('fuite totale', fuite_totale, flush=True)

for nom in noms:
    lots[nom][['chemin_relatif', 'sha256', 'etiquette']].to_csv(ART / f'split_{nom}.csv', index=False)
entrainement, validation, test = lots['entrainement'], lots['validation'], lots['test']
print('manifestes dedupliques ecrits dans', ART, flush=True)


In [ ]:
print('=== cellule : augmentation et pipeline de donnees ===', flush=True)
from PIL import Image, ImageEnhance, ImageOps


def augmenter_pil(image, graine):
    """Augmentation geometrique et photometrique, version independante."""
    alea = np.random.default_rng(graine)
    if alea.random() < 0.5:
        image = ImageOps.mirror(image)
    if alea.random() < 0.5:
        image = ImageOps.flip(image)
    angle = float(alea.uniform(-10.0, 10.0))
    if angle:
        image = image.rotate(angle, resample=Image.BILINEAR)
    largeur, hauteur = image.size
    facteur = float(alea.uniform(0.90, 1.00))
    cote = max(8, int(min(largeur, hauteur) * facteur))
    gauche = int(alea.integers(0, largeur - cote + 1))
    haut = int(alea.integers(0, hauteur - cote + 1))
    image = image.crop((gauche, haut, gauche + cote, haut + cote)).resize(
        (largeur, hauteur), Image.BILINEAR
    )
    image = ImageEnhance.Brightness(image).enhance(float(alea.uniform(0.90, 1.10)))
    image = ImageEnhance.Contrast(image).enhance(float(alea.uniform(0.90, 1.10)))
    return image


MODE_PIL = {'grayscale': 'L', 'rgb': 'RGB'}[COLOR_MODE]
_essai = augmenter_pil(Image.new(MODE_PIL, (16, 16), 128), 0)
assert _essai.size == (16, 16), _essai.size
assert _essai.mode == MODE_PIL, _essai.mode
print('augmentation PIL validee sur image factice', _essai.size, _essai.mode, flush=True)


def generer_entrainement(df):
    for indice, chemin in enumerate(df['chemin'].tolist()):
        etiquette = df['etiquette'].iloc[indice]
        image = keras.utils.load_img(chemin, target_size=IMG_SIZE, color_mode=COLOR_MODE)
        image = augmenter_pil(image, SEED + indice)
        yield keras.utils.img_to_array(image) / 255.0, float(etiquette)


def generer_simple(df):
    for chemin, etiquette in zip(df['chemin'].tolist(), df['etiquette'].tolist()):
        image = keras.utils.load_img(chemin, target_size=IMG_SIZE, color_mode=COLOR_MODE)
        yield keras.utils.img_to_array(image) / 255.0, float(etiquette)


AUTOTUNE = tf.data.AUTOTUNE
signature = (
    tf.TensorSpec(shape=IMG_SIZE + (1,), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.float32),
)
augmentation = keras.Sequential(
    [
        keras.layers.RandomFlip('horizontal'),
        keras.layers.RandomFlip('vertical'),
        keras.layers.RandomRotation(10.0 / 360.0),
        keras.layers.RandomZoom(0.10),
        keras.layers.RandomBrightness(0.10),
        keras.layers.RandomContrast(0.10),
    ],
    name='augmentation',
)
# Augmentation appliquee a chaque epoch, sans cache : figer une seule version
# augmentee par image est ce qui faisait surapprendre le modele des l'epoch 4.
ds_entrainement = (
    tf.data.Dataset.from_generator(lambda: generer_simple(entrainement), output_signature=signature)
    .shuffle(len(entrainement), seed=SEED, reshuffle_each_iteration=True)
    .map(lambda x, y: (augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)
ds_validation = (
    tf.data.Dataset.from_generator(lambda: generer_simple(validation), output_signature=signature)
    .cache()
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)
ds_test = (
    tf.data.Dataset.from_generator(lambda: generer_simple(test), output_signature=signature)
    .cache()
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)
for nom, ds in (('entrainement', ds_entrainement), ('validation', ds_validation), ('test', ds_test)):
    forme = tuple(next(iter(ds.take(1)))[0].shape[1:])
    print('lot', nom, 'signature', forme, flush=True)
    assert forme == IMG_SIZE + (1,), forme
print('pipeline pret', flush=True)


In [ ]:
print('=== cellule : architecture v1 relue depuis le .h5 ===', flush=True)
import h5py

CHEMIN_V1 = trouver_repertoire('aeris-modeles-v1') / 'modele_pneumonie.h5'
print('modele de reference :', CHEMIN_V1, flush=True)

with h5py.File(CHEMIN_V1, 'r') as f:
    config_brut = f.attrs['model_config']
    config_brut = config_brut if isinstance(config_brut, str) else config_brut.decode()
    formes = []
    f.visititems(
        lambda n, o: formes.append(o.shape)
        if isinstance(o, h5py.Dataset) and n.endswith('kernel:0') and o.ndim == 2 else None
    )
entree_dense_v1 = max(int(s[0]) for s in formes)
print('dimension d entree de la premiere Dense dans le .h5 :', entree_dense_v1, flush=True)

config = json.loads(config_brut)
if config['class_name'] != 'Sequential':
    raise SystemExit(f'classe de modele inattendue : {config["class_name"]}')
couches = config['config']['layers']
if couches[0]['class_name'] != 'InputLayer':
    couches = [{
        'class_name': 'InputLayer',
        'config': {
            'batch_input_shape': [None] + list(IMG_SIZE + (1,)),
            'dtype': 'float32',
            'sparse': False,
            'name': 'entree',
        },
        'name': 'entree',
        'inbound_nodes': None,
    }] + couches
    config['config']['layers'] = couches

# `dtype` est un artefact de serialisation de Keras 3 dans les initialiseurs ;
# il est sans effet sur Keras 2 et provoke un echec de deserialisation ailleurs.
INITIALISATEURS = (
    'VarianceScaling', 'GlorotUniform', 'GlorotNormal', 'RandomNormal',
    'RandomUniform', 'TruncatedNormal', 'Zeros', 'Ones', 'Constant', 'Identity',
)
def migrer(noeud):
    if isinstance(noeud, list):
        return [migrer(x) for x in noeud]
    if isinstance(noeud, dict):
        net = {k: migrer(v) for k, v in noeud.items()}
        classe = net.get('class_name')
        if classe in INITIALISATEURS and isinstance(net.get('config'), dict):
            net['config'].pop('dtype', None)
        return net
    return noeud


config = migrer(config)
v1 = keras.models.load_model(CHEMIN_V1, compile=False)
modele = keras.models.model_from_json(json.dumps(config))
print('v1 charge par tf_keras, v3 reconstruit depuis la meme config', flush=True)

# La perte doit suivre la forme de sortie de l'architecture relue dans le .h5 :
# la tuberculose sort un softmax a deux colonnes, la pneumonie une sigmoidule
# unique. Compiler la seconde avec sparse_categorical_crossentropy donne un
# `loss: nan` des le premier epoch.
if bool(False):
    perte = 'sparse_categorical_crossentropy'
else:
    perte = 'binary_crossentropy'
print('fonction de perte :', perte, flush=True)
modele.compile(
    optimizer=keras.optimizers.Adam(5e-4),
    loss=perte,
    metrics=['accuracy', keras.metrics.AUC(name='auc')],
)
print('v1 parametres', v1.count_params(), 'entree', v1.input_shape, 'sortie', v1.output_shape, flush=True)

noms_denses = [couche['config'].get('name') or couche['name'] for couche in couches
               if 'dense' in (couche['config'].get('name') or couche['name']).lower()]
assert noms_denses, 'aucune couche Dense dans la config'
premiere = modele.get_layer(noms_denses[0])
if int(premiere.kernel.shape[0]) != entree_dense_v1:
    raise SystemExit(
        f'ENTREE IMG_SIZE + (1,) incompatible avec le modele en service : la premiere '
        f'Dense attend {entree_dense_v1} entrees, le modele reconstruit en donne '
        f'{int(premiere.kernel.shape[0])}'
    )
if modele.count_params() != v1.count_params():
    raise SystemExit(
        f'nombre de parametres different : {modele.count_params()} reconstruit '
        f'contre {v1.count_params()}'
    )
if v1.count_params() == 0:
    raise SystemExit('v1 charge sans poids : la comparaison serait sans sens')
print('architecture identique au modele en service', flush=True)
print('entree', modele.input_shape, 'sortie', modele.output_shape, flush=True)
print('sortie a deux classes :', False, flush=True)


In [ ]:
print('=== cellule : entrainement ===', flush=True)
# val_loss oscille fortement d'un epoch a l'autre ; selectionner dessus
# revient a tirer au sort. On selectionne sur val_auc, la metrique du critere
# de remplacement, avec un LR plus bas pour stabiliser la validation.
rappels = [
    keras.callbacks.EarlyStopping(
        monitor='val_auc', mode='max', patience=15,
        restore_best_weights=True, min_delta=1e-5,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc', mode='max', factor=0.5, patience=6, min_lr=1e-6,
    ),
]

historique = modele.fit(
    ds_entrainement,
    validation_data=ds_validation,
    epochs=EPOCHS,
    callbacks=rappels,
    verbose=2,
)
print('epochs_effectives', len(historique.history['loss']), flush=True)
print('meilleure val_auc', max(historique.history.get('val_auc', [float('nan')])), flush=True)
pd.DataFrame(historique.history).to_csv(ART / 'courbe_apprentissage.csv', index=False)


In [ ]:
print('=== cellule : evaluation de v1 et v3 sur le meme hold-out ===', flush=True)
y_true = test['etiquette'].to_numpy(dtype=int)
print('hold-out', len(y_true), 'images,', int(y_true.sum()), 'positifs', flush=True)

def proba_positif(sortie, inverser):
    # Les deux pathologies n'ont pas la meme forme de sortie : la tuberculose
    # sort un softmax a deux colonnes dont la colonne 1 est la classe positive,
    # la pneumonie sort une sigmoidule unique. La convention du service est
    # inversee pour cette derniere (la sortie unique vaut P(NORMAL)), alors que
    # v3 est entraine ici avec l'etiquette 1 = PNEUMONIA et ne doit donc pas
    # etre inverse.
    if sortie.ndim == 2 and sortie.shape[1] == 2:
        return sortie[:, 1]
    plat = sortie.reshape(-1)
    return 1.0 - plat if inverser else plat


sortie_v3 = modele.predict(ds_test, verbose=0)
if (sortie_v3.ndim == 2 and sortie_v3.shape[1] == 2) != bool(False):
    raise SystemExit(
        f'v3 sort {sortie_v3.shape[-1]} colonne(s) alors que la configuration en '
        f'attend {2 if bool(False) else 1}'
    )
proba_v3 = proba_positif(sortie_v3, False)
print('v3 sortie', sortie_v3.shape, '| P(positif) directe, etiquetage local', flush=True)

print('v1 deja charge et weights lus :', v1.count_params(), 'parametres', flush=True)
print('entree v1', v1.input_shape, 'sortie v1', v1.output_shape, flush=True)
if tuple(v1.input_shape[1:]) != IMG_SIZE + (1,):
    raise SystemExit(f'specification v1 inattendue : {v1.input_shape}')

sortie_v1 = v1.predict(ds_test, verbose=0)
proba_v1 = proba_positif(sortie_v1, True)
if sortie_v1.ndim == 2 and sortie_v1.shape[1] == 2:
    polarite_v1 = 'softmax colonne 1'
elif True:
    polarite_v1 = 'sortie unique inversee (convention service : P(NORMAL))'
else:
    polarite_v1 = 'sortie unique directe'
print('v1 sortie', sortie_v1.shape, '|', polarite_v1, flush=True)


# Bornes partagees avec le tableau de bord (src/lib/format.ts) : au-dela de
# 0.995 l'affichage dit 'plus de 99 %', jamais '100 %'. Toute sortie de modele
# doit rester sous ce plafond, calibration comprise.
SEUIL_SATURATION = 0.995


def vers_logit(proba):
    p = np.clip(np.asarray(proba, dtype=float), EPS, 1.0 - EPS)
    return np.log(p / (1.0 - p))


def temperature_optimal(logits, y, eps=1e-12):
    z = np.clip(np.asarray(logits, dtype=float), -60.0, 60.0)
    y = np.asarray(y, dtype=float)

    # Calibration de temperature sur un classifieur binaire : la sigmoidule
    # divisee par t, pas un softmax. Ecrire ici un softmax normalise sur
    # l'axe des exemples donnerait une fonction de cout constante en t, et
    # une temperature sans signification.
    def cout(t):
        p = 1.0 / (1.0 + np.exp(-z / t))
        p = np.clip(p, eps, 1.0 - eps)
        return -float(np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

    grille = np.linspace(0.05, 50.0, 400)
    couts = np.array([cout(t) for t in grille])
    meilleur = int(np.argmin(couts))
    if meilleur == 0 or meilleur == len(grille) - 1:
        return None, 'optimum a la frontiere du balayage'
    a, b = grille[meilleur - 1], grille[meilleur + 1]
    for _ in range(80):
        m1 = a + (b - a) / 3.0
        m2 = b - (b - a) / 3.0
        if cout(m1) < cout(m2):
            b = m2
        else:
            a = m1
    return float((a + b) / 2.0), 'estimable'


def point_a_rappel(y, proba, rappel_cible=0.90):
    ordre = np.argsort(-proba)
    y_tri = y[ordre]
    cumul = np.cumsum(y_tri)
    total = y.sum()
    idx = np.searchsorted(cumul, rappel_cible * total)
    idx = min(idx, len(y) - 1)
    seuil = float(proba[ordre][idx])
    predits = proba >= seuil
    vp = int(np.sum(predits & (y == 1)))
    fp = int(np.sum(predits & (y == 0)))
    fn = int(np.sum((~predits) & (y == 1)))
    return {
        'seuil': seuil,
        'rappel': vp / total if total else None,
        'precision': vp / (vp + fp) if (vp + fp) else None,
        'specificite': (int(np.sum((~predits) & (y == 0))) / int(np.sum(y == 0))) if np.sum(y == 0) else None,
        'VP': vp, 'FP': fp, 'FN': fn,
    }


def mesurer(nom, proba):
    logits = vers_logit(proba)
    t, statut = temperature_optimal(logits, y_true)
    resultat = {
        'modele': nom,
        'polarite': polarite_v1 if nom.startswith('v1') else 'sortie du modele',
        'auc': float(roc_auc_score(y_true, proba)),
        'brier': float(brier_score_loss(y_true, np.clip(proba, 0.0, 1.0))),
        'nll': float(log_loss(y_true, np.clip(proba, EPS, 1 - EPS))),
        'saturation_float32': int(np.sum(np.asarray(proba) >= 1.0 - EPS)),
        'affichait_100_pourcent': int(np.sum(np.round(proba, 4) >= 0.995)),
        'temperature': t,
        'temperature_statut': statut,
        'point_rappel_090': point_a_rappel(y_true, proba),
    }
    if t is None:
        t = 1.0
        statut = f'{statut} ; temperature neutre retenue'
    p_cal = 1.0 / (1.0 + np.exp(-logits / t))
    # Plafond dur : l'affichage doit porter un pourcentage credible, jamais
    # 100 %. Le plafond est aligne sur SEUIL_SATURATION du tableau de bord,
    # afin qu'une probabilite servie egale a 0.995 se lise 'plus de 99 %'.
    # La transformation est monotone : elle ne change pas le classement, donc
    # pas l'AUC ni le point a rappel 0.90.
    p_fin = np.clip(p_cal, EPS, SEUIL_SATURATION)
    resultat['calibration'] = {
        'temperature': t,
        'temperature_statut': statut,
        'plafond': SEUIL_SATURATION,
        'proba_max': float(np.max(p_fin)),
        'brier': float(brier_score_loss(y_true, p_fin)),
        'nll': float(log_loss(y_true, p_fin)),
        'point_rappel_090': point_a_rappel(y_true, p_fin),
        'affichait_100_pourcent': int(np.sum(np.round(p_fin, 4) >= SEUIL_SATURATION + 1e-9)),
    }
    return resultat


pd.DataFrame({
    'chemin_relatif': test['chemin_relatif'].to_numpy(),
    'etiquette': y_true,
    'proba_v1': proba_v1,
    'proba_v3': proba_v3,
}).to_csv(ART / 'predictions_test_pn_edge_v3.csv', index=False)

m_v1 = mesurer('v1 (en service)', proba_v1)
m_v3 = mesurer('v3', proba_v3)
rapport = {
    'hold-out': {
        'taille': int(len(y_true)),
        'positifs': int(y_true.sum()),
        'deduplique_sha256': True,
        'fuite_inter_lots': 0,
        'construction': 'split par regroupement de contenu SHA-256, 80/10/10 par classe',
    },
    'v1': m_v1,
    'v3': m_v3,
    'v3_details': {
        'architecture': 'identique a v1',
        'parametres': int(modele.count_params()),
        'epochs_effectifs': len(historique.history['loss']),
        'nb_entrainement': int(len(entrainement)),
        'nb_validation': int(len(validation)),
    },
    'remplacements': {
        'auc': bool(m_v3['auc'] > m_v1['auc']),
        'brier': bool(m_v3['brier'] < m_v1['brier']),
        'nll': bool(m_v3['nll'] < m_v1['nll']),
        'point_090_binaire': bool(
            (m_v3['point_rappel_090']['FP'] + m_v3['point_rappel_090']['FN'])
            < (m_v1['point_rappel_090']['FP'] + m_v1['point_rappel_090']['FN'])
        ),
    },
}
(ART / 'metrics_pn_edge_v3.json').write_text(json.dumps(rapport, indent=2))
# Deux formats : .keras pour tf_keras (source), .h5 pour les services
# en tensorflow 2.15/2.16, dont tensorflow.keras.models.load_model
# ne connait pas le conteneur .keras.
modele.save(ART / 'aeris_pn_edge_v3.keras')
modele.save(ART / 'aeris_pn_edge_v3.h5')

print('', flush=True)
print('==== RESULTATS ===', flush=True)
for cle in ('auc', 'brier', 'nll', 'saturation_float32', 'affichait_100_pourcent', 'temperature'):
    print(f"{cle:26} v1={m_v1[cle]!s:12} v3={m_v3[cle]!s}", flush=True)
print('v1 point@0.90', json.dumps(m_v1['point_rappel_090']), flush=True)
print('v3 point@0.90', json.dumps(m_v3['point_rappel_090']), flush=True)
print('remplacements', json.dumps(rapport['remplacements']), flush=True)
